<a href="https://colab.research.google.com/github/kaouchounesalah-eddine-ux/arabic-news-classification/blob/main/05_base_arabic_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Load dataset & Keep final test set

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd

from sklearn.model_selection import train_test_split
# from preprocess import preprocess


# =========================
# 1. Load dataset
# =========================

import pandas as pd

df = pd.read_json(
    '/content/drive/MyDrive/articles.json'
)

X = df["body"]
y = df["categories"].str[0]


# =========================
# 2. Keep final test set
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


2. Development Data

In [5]:
print("=== DEVELOPMENT DATA ===")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nNumber of samples:", len(X_train))

print("\nCategory distribution:")
print(y_train.value_counts())

=== DEVELOPMENT DATA ===
X_train shape: (16800,)
y_train shape: (16800,)

Number of samples: 16800

Category distribution:
categories
ثقافة     2800
دولي      2800
اقتصاد    2800
رياضة     2800
سياسة     2800
مجتمع     2800
Name: count, dtype: int64


3. Preprocessing

In [6]:
import unicodedata


def normalize_arabic(text):
    return (
        text.replace("أ", "ا")
            .replace("إ", "ا")
            .replace("آ", "ا")
    )


def remove_diacritics(text):
    return "".join(
        char
        for char in text
        if unicodedata.category(char) != "Mn"

    )


def remove_tatweel(text):
    return text.replace("ـ", "")

def remove_taarif(text):
    return " ".join(
        word.removeprefix("ال")
        for word in text.split()
    )

def remove_numbers(text):
    return "".join(
        char
        for char in text
        if not char.isdigit()
    )

def remove_marks(text):
    return text.replace('\\"', '"').replace('"', '') if isinstance(text, str) else text

arabic_stopwords = {
    "في", "من", "إلى", "على", "عن", "و", "أو",
    "أن", "إن", "كان", "كانت", "هذا", "هذه",
    "ذلك", "التي", "الذي", "هو", "هي", "هم",
    "ما", "لا", "لم", "لن", "مع", "كما"
}

def remove_stopwords(text):
    return " ".join(
        word
        for word in text.split()
        if word not in arabic_stopwords
    )

def preprocess(text):
    text = normalize_arabic(text)
    text = remove_diacritics(text)
    text = remove_tatweel(text)
    text = remove_taarif(text)
    text = remove_numbers(text)
    text = remove_marks(text)
    text = remove_stopwords(text)

    return text

In [7]:
X_train = X_train.apply(preprocess)
X_test = X_test.apply(preprocess)

4. X_trainEmbedding

In [8]:
from sentence_transformers import SentenceTransformer

arabic_embedding_model = SentenceTransformer(
    "Omartificial-Intelligence-Space/mmbert-base-arabic-nli"
)

modules.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/13.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/61.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/48.5k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

In [9]:
print("Arabic embedding model loaded.")
test_embeddings_ar = arabic_embedding_model.encode(
    X_train.iloc[:5].tolist()
)

print("Embeddings shape:", test_embeddings_ar.shape)

Arabic embedding model loaded.
Embeddings shape: (5, 768)


In [10]:
X_train_embeddings_ar = arabic_embedding_model.encode(
    X_train.tolist(),
    show_progress_bar=True,
    batch_size=32
)

print("Embeddings shape:", X_train_embeddings_ar.shape)

Batches:   0%|          | 0/525 [00:00<?, ?it/s]

Embeddings shape: (16800, 768)


In [11]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_scores_ar = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_train_embeddings_ar, y_train),
    start=1
):

    X_fold_train = X_train_embeddings_ar[train_idx]
    X_fold_val = X_train_embeddings_ar[val_idx]

    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    model = LogisticRegression(
        C=1.0,
        max_iter=5000
    )

    model.fit(X_fold_train, y_fold_train)

    y_fold_pred = model.predict(X_fold_val)

    fold_f1 = f1_score(
        y_fold_val,
        y_fold_pred,
        average="macro"
    )

    fold_scores_ar.append(fold_f1)

    print(
        f"Fold {fold}: "
        f"Macro F1 = {fold_f1:.4f}"
    )



Fold 1: Macro F1 = 0.8320
Fold 2: Macro F1 = 0.8388
Fold 3: Macro F1 = 0.8432
Fold 4: Macro F1 = 0.8252
Fold 5: Macro F1 = 0.8384


In [12]:
print("\nbase-arabic Embedding + Logistic Regression")

print(
    f"Mean Macro F1: "
    f"{np.mean(fold_scores_ar):.4f}"
)

print(
    f"Standard deviation: "
    f"{np.std(fold_scores_ar):.4f}"
)


base-arabic Embedding + Logistic Regression
Mean Macro F1: 0.8355
Standard deviation: 0.0063


 X_test embeddings

In [15]:
X_test_embeddings_ar = arabic_embedding_model.encode(
    X_test.tolist(),
    batch_size=32,
    show_progress_bar=True
)

Batches:   0%|          | 0/132 [00:00<?, ?it/s]

Final test

In [16]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import pandas as pd

# 1. Train on ALL training embeddings
final_model_ar = LogisticRegression(
    C=1.0,
    max_iter=5000
)

final_model_ar.fit(
    X_train_embeddings_ar,
    y_train
)

print("Final model trained!")

# 2. Predict the held-out test set
y_test_pred_ar = final_model_ar.predict(
    X_test_embeddings_ar
)

# 3. Calculate evaluation metrics
accuracy = accuracy_score(
    y_test,
    y_test_pred_ar
)

macro_f1 = f1_score(
    y_test,
    y_test_pred_ar,
    average="macro"
)

print("\n===== FINAL TEST RESULTS =====")
print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")

# 4. Show precision, recall, and F1 for each category
print("\n===== CLASSIFICATION REPORT =====")
print(
    classification_report(
        y_test,
        y_test_pred_ar,
        zero_division=0
    )
)

# 5. Confusion matrix
labels = sorted(y_test.unique())

cm = confusion_matrix(
    y_test,
    y_test_pred_ar,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

print("\n===== CONFUSION MATRIX =====")
print("Rows = actual labels; columns = predicted labels")
display(cm_df)

Final model trained!

===== FINAL TEST RESULTS =====
Accuracy: 0.8507
Macro F1: 0.8502

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

      اقتصاد       0.80      0.80      0.80       700
       ثقافة       0.88      0.92      0.90       700
        دولي       0.84      0.86      0.85       700
       رياضة       0.99      0.99      0.99       700
       سياسة       0.81      0.79      0.80       700
       مجتمع       0.77      0.75      0.76       700

    accuracy                           0.85      4200
   macro avg       0.85      0.85      0.85      4200
weighted avg       0.85      0.85      0.85      4200


===== CONFUSION MATRIX =====
Rows = actual labels; columns = predicted labels


,اقتصاد,ثقافة,دولي,رياضة,سياسة,مجتمع
اقتصاد,557,28,18,1,47,49
ثقافة,25,641,7,2,10,15
دولي,17,17,602,2,30,32
رياضة,0,0,2,694,2,2
سياسة,41,15,33,0,555,56
مجتمع,56,25,51,2,42,524


In [18]:
import joblib

joblib.dump(final_model_ar, "embedding_classifier_ar.joblib")

['embedding_classifier_ar.joblib']

In [21]:
from huggingface_hub import login

login()

In [22]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_file(
    path_or_fileobj="embedding_classifier_ar.joblib",
    path_in_repo="embedding_classifier_ar.joblib",
    repo_id="salah-2005/base-arabic_embeddings",
    repo_type="model"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ding_classifier_ar.joblib: 100%|##########| 38.0kB / 38.0kB            

CommitInfo(commit_url='https://huggingface.co/salah-2005/base-arabic_embeddings/commit/39ceacec8f2b8f0db3ca487336145f718d921f2b', commit_message='Upload embedding_classifier_ar.joblib with huggingface_hub', commit_description='', oid='39ceacec8f2b8f0db3ca487336145f718d921f2b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/salah-2005/base-arabic_embeddings', endpoint='https://huggingface.co', repo_type='model', repo_id='salah-2005/base-arabic_embeddings'), pr_revision=None, pr_num=None)

In [23]:
import joblib
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer

repo_id = "salah-2005/base-arabic_embeddings"

classifier_path = hf_hub_download(
    repo_id=repo_id,
    filename="embedding_classifier_ar.joblib"
)

classifier = joblib.load(classifier_path)

embedding_model = SentenceTransformer(
    "Omartificial-Intelligence-Space/mmbert-base-arabic-nli"
)

embedding_classifier_ar.joblib: reconstructing file:   0%|          |  0.00B / 38.0kB            

embedding_classifier_ar.joblib: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]